# image_flux2_klein

**FLUX.2 klein** 으로 World Reference → 장소 배경 → 인물 초상.

| | |
|---|---|
| 커널 | `flux` |
| 출력 | `outputs/image_flux2_klein/<시나리오>/` |

---

## 실행 전 반드시 읽을 것

### 1. GPU 세대 요구가 있다

klein의 텍스트 인코더는 `Qwen/Qwen3-4B-FP8` 이다.
**FP8 연산은 compute capability 8.9 이상에서만 동작한다.**

| GPU | CC | 가능 |
|---|---|---|
| RTX 3090 / A100 | 8.6 / 8.0 | **불가** |
| RTX 4090 / L40S | 8.9 | 가능 |
| H100 / H200 | 9.0 | 가능 |
| RTX 5090 | 12.0 | 가능 |

3090에서 돌리면 아래 에러로 멈춘다. 버전 문제가 아니라 하드웨어 문제라 우회가 없다.

```
ValueError: FP8 quantized models is only supported on GPUs with
compute capability >= 8.9 (e.g 4090/H100), actual = 8.6
```

3번 셀이 이걸 먼저 확인하고, 안 되면 가중치를 받기 전에 멈춘다.

### 2. diffusers 경로는 쓸 수 없다

`diffusers.Flux2Pipeline` 은 FLUX.2 **[dev]** 용으로 만들어졌고 Mistral 인코더를 기대한다.
klein은 Qwen3를 쓰기 때문에 채팅 템플릿 렌더링에서 터진다.

```
Expected types for text_encoder: Mistral3ForConditionalGeneration,
got Qwen3ForCausalLM
TypeError: can only concatenate str (not "list") to str
```

그래서 **공식 저장소(github.com/black-forest-labs/flux2)의 CLI를 통해 돌린다.**

### 3. 라이선스

- **klein 4B — Apache 2.0.** 상업 이용 가능
- **klein 9B — 비상업 라이선스.** 서비스에 쓸 수 없다

기본값을 4B로 두었다. 9B로 바꾸면 셀이 경고한다.


## 1. 커널 확인

설치와 저장소 clone 은 `setup_kernels.ipynb` 가 한다. 커널을 `flux` 로 바꿀 것.

In [ ]:
# ---------------------------------------------------------------------------
# 커널 확인. 설치는 setup_kernels.ipynb 가 담당한다.
#
# 이 셀에 pip install 을 추가하지 말 것. 노트북 4개는 서로 양립 불가능한
# torch/transformers 를 쓰기 때문에, 여기서 다시 깔면 커널 조합이 깨진다.
#   ace  torch 2.10.0+cu128 / transformers >=4.51,<4.58
#   sa3  torch 2.7.1 cu126  / transformers >=5.8
#   flux torch 2.8.0        / transformers ==4.56.1
# ---------------------------------------------------------------------------
import sys, importlib

KERNEL = "flux"
NEED   = ["torch", "transformers", "PIL", "fire"]

print("python     :", sys.version.split()[0])
print("interpreter:", sys.executable)
print()

missing = []
for m in NEED:
    try:
        mod = importlib.import_module(m)
        print(f"OK   {m:16s} {getattr(mod, '__version__', '')}")
    except Exception as e:
        missing.append(m)
        print(f"FAIL {m:16s} {type(e).__name__}: {e}")

assert not missing, (
    f"\n{missing} 를 import 하지 못했다.\n"
    f"커널을 '{KERNEL}' 로 바꿀 것 (Kernel > Change kernel).\n"
    f"'{KERNEL}' 커널이 목록에 없으면 setup_kernels.ipynb 를 먼저 실행한다.\n"
    f"여기서 pip install 로 때우지 말 것 — 버전 조합이 깨진다."
)
print("\n커널 OK")


import os, pathlib

# setup_kernels 가 만든 커널이면 venv 가 저장소 안에 있다: <repo>/.venv
FLUX_DIR = pathlib.Path(os.environ.get("FLUX_DIR", pathlib.Path(sys.prefix).parent))
assert (FLUX_DIR / "scripts" / "cli.py").exists(), (
    f"flux2 저장소를 못 찾았다: {FLUX_DIR}\n"
    f"FLUX_DIR 환경변수로 지정하거나 setup_kernels.ipynb 를 먼저 실행할 것."
)
print("flux2 저장소:", FLUX_DIR)

## 2. 인증

klein 가중치와 Qwen3 인코더를 받으려면 HF 토큰이 필요하다.


In [ ]:
import os, pathlib

# 토큰을 노트북에 적지 않는다. 커밋되면 그대로 노출된다.
# 터미널에서:  export HF_TOKEN=hf_xxx   (커널을 띄우기 전에. 이미 떠 있으면 커널 재시작)
# 또는 파일로:  echo hf_xxx > ~/.hf_token
tok = os.environ.get("HF_TOKEN", "").strip()
if not tok:
    p = pathlib.Path.home() / ".hf_token"
    if p.exists():
        tok = p.read_text().strip()
        os.environ["HF_TOKEN"] = tok

assert tok.startswith("hf_"), (
    "HF_TOKEN 이 없다.\n"
    "  export HF_TOKEN=hf_...   후 커널 재시작\n"
    "  또는  echo hf_... > ~/.hf_token\n"
    "gated repo 라 토큰 없이는 401 이 난다."
)
print("토큰 설정됨:", tok[:6] + "…" + f" ({len(tok)}자)")

## 3. GPU 확인

**여기서 막히면 이 노트북은 이 장비에서 못 돌린다.** 가중치를 받기 전에 멈춘다.


In [ ]:
import torch

assert torch.cuda.is_available(), "GPU가 없다. 계산 노드에서 실행할 것."

p = torch.cuda.get_device_properties(0)
cc = (p.major, p.minor)
print(f"GPU  : {p.name}")
print(f"VRAM : {round(p.total_memory / 1e9, 1)} GB")
print(f"CC   : {p.major}.{p.minor}")

if cc < (8, 9):
    raise SystemExit(
        f"\nFP8 미지원 세대다 (CC {p.major}.{p.minor} < 8.9).\n"
        f"klein의 텍스트 인코더가 Qwen3-4B-FP8 이라 이 GPU에서는 못 돌린다.\n"
        f"4090 / L40S / H100 급으로 옮기거나, image_hidream 노트북을 쓸 것."
    )

print("\nFP8 지원 확인. 진행 가능.")


## 4. 설정


In [ ]:
import os, sys, json, pathlib, getpass, shutil

USER = getpass.getuser()

# ---- 이 노드에서 유효한 루트를 직접 찾는다 --------------------------------
# 학과 클러스터는 같은 저장소가 노드마다 다른 경로에 마운트된다.
#   로그인 노드(hpcmaster) : /data1/$USER   (= HOME)
#   계산 노드(hpc-stat1)   : /mnt/data1/$USER
# setup_kernels 로 만든 커널이면 PROJ_BASE 가 kernel.json 에 박혀 있고,
# Slurm 으로 돌리면 common.sh 가 환경변수로 넘겨준다. 둘 다 아니면 여기서 찾는다.
def pick_root():
    for c in (f"/mnt/data1/{USER}", f"/data1/{USER}",
              str(pathlib.Path.home()), "/workspace", "/scratch"):
        if pathlib.Path(c).is_dir() and os.access(c, os.W_OK):
            return c
    return None

ROOT = os.environ.get("PROJ_ROOT") or pick_root()
if ROOT is None:
    raise SystemExit(f"쓸 수 있는 루트를 못 찾았다 (user={USER})")

# 계산 노드에는 $HOME 이 아예 없다. 두면 캐시/kernelspec 이 전부 깨진다.
if not pathlib.Path(os.environ.get("HOME", "/nonexistent")).is_dir():
    os.environ["HOME"] = ROOT

BASE  = pathlib.Path(os.environ.get("PROJ_BASE") or f"{ROOT}/mystery")
MODEL = "image_flux2_klein"
OUT_DIR = BASE / "outputs" / MODEL

# 캐시는 홈이 아니라 대용량 스토리지로. 홈 쿼터면 다운로드 중 os error 28 이 난다.
os.environ.setdefault("HF_HOME",      str(BASE / "hf_cache"))
os.environ.setdefault("TMPDIR",       str(BASE / "tmp"))
os.environ.setdefault("MPLCONFIGDIR", str(BASE / "mpl_cache"))

try:
    for p in (OUT_DIR, pathlib.Path(os.environ["HF_HOME"]),
              pathlib.Path(os.environ["TMPDIR"])):
        p.mkdir(parents=True, exist_ok=True)
except (PermissionError, OSError) as e:
    raise SystemExit(
        f"{BASE} 에 쓸 수 없다 -> {type(e).__name__}: {e}\n"
        f"  이 셀 맨 위에서 지정할 것:\n"
        f"      os.environ['PROJ_BASE'] = '/data1/{USER}/mystery'"
    )

# ---- 입력 JSON (scenarios_en/*.json 전부) ---------------------------------
# papermill 로 돌리면 cwd 가 /tmp 라 "." 만으로는 못 찾는다.
# PROJ_INPUT(환경변수) 을 먼저 보고, 없으면 후보들을 뒤진다.
def pick_input_dir():
    cands = []
    if os.environ.get("PROJ_INPUT"):
        cands.append(pathlib.Path(os.environ["PROJ_INPUT"]))
    cands += [pathlib.Path.cwd(), pathlib.Path.cwd().parent, BASE]
    for c in cands:
        if (c / "scenarios_en").is_dir() and list((c / "scenarios_en").glob("*.json")):
            return c
    # 마지막 수단: ROOT 아래를 얕게 훑는다
    for c in pathlib.Path(ROOT).glob("*/*/"):
        if (c / "scenarios_en").is_dir() and list((c / "scenarios_en").glob("*.json")):
            return c
    return None

IN_DIR = pick_input_dir()
if IN_DIR is None:
    raise SystemExit(
        "scenarios_en/*.json 을 못 찾았다.\n"
        f"  PROJ_INPUT = {os.environ.get('PROJ_INPUT', '(미설정)')}\n"
        f"  cwd        = {pathlib.Path.cwd()}\n"
        f"  이 셀 맨 위에서:  os.environ['PROJ_INPUT'] = '/data1/{USER}/<저장소>/Jaemin'"
    )

# scenario_id(파일 내부 필드, 예: "12_heungbu")를 키로 쓴다 — 파일명과 무관하게 안정적.
SCENARIO_FILES = sorted((IN_DIR / "scenarios_en").glob("*.json"))
assert SCENARIO_FILES, f"scenarios_en 에 json 이 없다: {IN_DIR / 'scenarios_en'}"

DATA = {}
for _f in SCENARIO_FILES:
    _d = json.load(open(_f, encoding="utf-8"))
    DATA[_d["scenario_id"]] = _d

for k in DATA:
    (OUT_DIR / k).mkdir(parents=True, exist_ok=True)


def out_path(scen, *parts):
    """OUT_DIR/<시나리오>/<...> 경로를 만들고 돌려준다."""
    p = OUT_DIR / scen
    for x in parts[:-1]:
        p = p / x
    p.mkdir(parents=True, exist_ok=True)
    return p / parts[-1]


print("node    :", os.uname().nodename)
print("ROOT    :", ROOT)
print("BASE    :", BASE, f"(여유 {shutil.disk_usage(BASE).free/1e9:.0f} GB)")
print("IN_DIR  :", IN_DIR)
print("HF_HOME :", os.environ["HF_HOME"])
print("OUT_DIR :", OUT_DIR)
print(f"시나리오 {len(DATA)}개:")
for k, d in DATA.items():
    print(f"  {k:24s} 장소 {len(d['places'])} / 인물 {len(d['cast'])}")


## 5. 프롬프트 조립

`image_hidream` 과 동일하다. 같은 프롬프트를 넣어야 모델 비교가 성립한다.


In [ ]:
# 두 이미지 노트북(image_hidream / image_flux2_klein)에 동일하게 들어간다.
# 프롬프트가 달라지면 모델 비교가 성립하지 않는다.
#
# STYLE: HiDream 쪽 파이프라인은 레퍼런스 이미지를 받지 못한다
#        (HiDreamImagePipeline.__call__ 에 image 인자가 없다).
#        그래서 세계관 앵커를 문장으로 고정한다. FLUX 는 이 문장 + world_ref
#        이미지를 함께 쓴다.
STYLE = ("cohesive painterly realism, muted desaturated palette, "
         "soft directional daylight, film grain, no text, no watermark")


def build_prompts(d):
    era      = d["setting"]["era"]
    loc      = d["setting"]["location"]
    tone     = d["adapted_story"]["tone"]
    synopsis = d["adapted_story"]["synopsis"]
    palette  = d["adapted_story"]["palette"]

    world = (f"empty, uninhabited establishing reference image — no people, "
             f"no humans, no figures, no bodies, no corpse, nobody present "
             f"anywhere in frame. defines the visual world. "
             f"period and place: {era}, {loc}. mood: {tone}. "
             f"color palette: {palette}. consistent color palette, lighting "
             f"logic and material language. completely vacant, deserted, "
             f"empty of any person or body no matter what else is implied. "
             f"{STYLE}.")

    places = {}
    for p in d["places"]:
        clue_list = "; ".join(p["clues"])
        places[p["place_id"]] = (
            f"empty, uninhabited space — no people, no humans, no figures, "
            f"no bodies, no corpse, nobody present anywhere in frame. "
            f"{p['name']}. {p['description']} "
            f"this scene must clearly show: {clue_list} — as objects and "
            f"traces only, with no person or body present to go with them. "
            f"period and place: {era}, {loc}. mood: {tone}. "
            f"color palette: {palette}. "
            f"completely vacant, deserted, unoccupied, empty of any person "
            f"or body no matter what else is implied. {STYLE}.")

    # 이름은 절대 넣지 않는다. 외모 묘사만으로 그린다.
    portraits = {c["id"]: (f"portrait of a {c['age']}-year-old {c['gender']}, "
                           f"{c['occupation']}. {c['appearance']} "
                           f"context: {synopsis} "
                           f"setting: {era}. mood: {tone}. "
                           f"color palette: {palette}. "
                           f"waist-up framing, neutral background, even lighting. "
                           f"{STYLE}.")
                 for c in d["cast"]}
    return {"world": world, "places": places, "portraits": portraits}


P = {k: build_prompts(d) for k, d in DATA.items()}

# 시나리오마다 자기 source_title 단어를 금지어로 쓴다 — 하드코딩된 2개짜리
# 리스트 대신, 20개 전부에서 원작 고유명사가 새어 들어갔는지 자동으로 잡는다.
STUDIO_TERMS = ["disney", "pixar", "dreamworks", "ghibli"]

STOPWORDS = {
    "the", "a", "an", "of", "in", "on", "for", "with", "to", "and", "or",
    "korean", "folktale", "fairy", "tale", "tales", "myth", "legend", "legends",
    "novel", "classical", "pansori", "based", "original", "setting", "custom",
    "little", "big", "great", "old", "young", "new",
    "girl", "boy", "man", "woman", "king", "queen", "prince", "princess",
    "red", "white", "black", "gold", "silver", "snow",
    "hood", "riding", "match", "sun", "moon", "woodcutter",
    "night", "day", "house", "story",
    # 일반 명사인데 우연히 제목에 들어간 것들 (실제 IP 고유명사가 아님)
    "piper", "office", "startup", "theater", "theatre", "silent-film", "silent", "film",
}

def banned_terms(d):
    import re
    title = re.sub(r"\(.*?\)", "", d.get("source_title", ""))
    words = {w.strip(".,").lower() for w in title.split() if len(w) > 2}
    return (words - STOPWORDS) | set(STUDIO_TERMS)

import re

bad = []
for scen, v in P.items():
    blob = " ".join([v["world"]] + list(v["places"].values())
                    + list(v["portraits"].values())).lower()
    # 단어 경계로 검사한다 — "pied" 가 "unoccupied" 안의 부분문자열로 잡히는 식의
    # 오탐(substring match)을 막는다.
    hits = [b for b in banned_terms(DATA[scen])
            if re.search(rf"\b{re.escape(b)}\b", blob)]
    if hits:
        bad.append((scen, hits))

for scen, v in P.items():
    print("=" * 70); print(scen)
    print("  [world]", v["world"][:120], "...")
    print(f"  장소 {len(v['places'])} / 인물 {len(v['portraits'])}")

print()
print("이름 유입 검사:", "OK" if not bad else f"FAIL {bad}")
assert not bad, "ENRICH 의 occupation / location / place name 을 확인할 것"
total_places = sum(len(v["places"]) for v in P.values())
total_portraits = sum(len(v["portraits"]) for v in P.values())
print(f"생성 예정: world {len(P)}장 + 장소 {total_places}장 + 인물 {total_portraits}장 "
      f"= {len(P) + total_places + total_portraits}장")

## 6. CLI 인터페이스 확인

공식 저장소는 `scripts/cli.py` 를 진입점으로 쓴다. **인자 이름이 버전마다 바뀔 수 있으므로
먼저 확인한다.** 아래 출력과 7번 셀의 `build_args()` 가 어긋나면 그 함수만 고치면 된다.


In [ ]:
import subprocess, os, sys

# 공식 진입점은 fire 기반이다:
#   main(model_name=None, single_eval=False, prompt=None,
#        debug_mode=False, cpu_offloading=False, **overwrite)
# 나머지 값(width/height/num_steps/guidance/seed/input_images)은 **overwrite 로
# 들어가 Config 에 setattr 된다. Config 에 없는 키는 "! unknown key" 로 무시된다.
env = dict(os.environ, PYTHONPATH=str(FLUX_DIR / "src"))
r = subprocess.run([sys.executable, str(FLUX_DIR / "scripts" / "cli.py"), "--help"],
                   cwd=FLUX_DIR, env=env, capture_output=True, text=True)
print(r.stdout[-3000:])
print(r.stderr[-1500:])
print("""
Config 필드 (이 이름만 --key value 로 넘길 수 있다):
  prompt  seed  width  height  num_steps  guidance
  input_images  match_image_size  upsample_prompt_mode  openrouter_model
--output_dir 같은 건 없다. CLI 는 <실행 디렉터리>/output/sample_N.png 에 저장한다.
""")

## 7. 생성 함수

CLI를 subprocess로 부른다. 파이썬 API를 직접 호출하는 방법도 있지만,
`src/flux2/` 에 `__init__.py` 가 없고 샘플링 루프를 직접 짜야 해서 CLI 쪽이 안전하다.

**모델 하나 로딩에 시간이 걸리므로, 한 번 띄워놓고 여러 장 뽑는 것이 훨씬 빠르다.**
아래는 한 장씩 부르는 단순한 버전이다. 장수가 많아지면 `--loop` 모드로 바꿀 것.


In [ ]:
import subprocess, sys, shutil, tempfile, pathlib, os

MODEL_NAME = "flux.2-klein-4b"     # 4B = Apache-2.0. 9B는 비상업이라 서비스에 못 쓴다
NUM_STEPS  = 4                     # klein은 4-step distilled
GUIDANCE   = 1.0                   # klein은 distilled라 guidance=1.0 고정, 다른 값 주면 ValueError
SIZE       = 1024

if "9b" in MODEL_NAME:
    print("경고: klein 9B는 비상업 라이선스다. 상용 서비스에는 4B만 쓸 수 있다.")


def build_args(prompt, refs=None, seed=0, size=SIZE):
    """scripts/cli.py 는 Fire(main) 이라 main() 의 파라미터명이 곧 플래그다.

        main(model_name, single_eval, prompt, debug_mode, cpu_offloading, **overwrite)

    width/height/num_steps/guidance/seed/input_images 는 **overwrite 로 들어가
    Config 필드를 덮어쓴다.
    """
    args = [
        sys.executable, str(FLUX_DIR / "scripts" / "cli.py"),
        f"--model_name={MODEL_NAME}",
        "--single_eval=True",          # 없으면 대화형 루프로 빠져 아무것도 안 나온다
        "--prompt", prompt,
        f"--width={size}",
        f"--height={size}",
        f"--num_steps={NUM_STEPS}",
        f"--guidance={GUIDANCE}",
        f"--seed={seed}",
        # 48GB 카드에서 텍스트 인코더 + 트랜스포머를 동시에 올리면 스모크 테스트
        # 첫 호출에서부터 OOM 난다 (47.4GB 사용 중에 320MiB 못 늘림).
        # cpu_offloading 을 켜면 CLI 가 둘을 번갈아 CPU<->GPU 로 옮긴다.
        "--cpu_offloading=True",
    ]
    if refs:
        args.append("--input_images=" + ",".join(str(pathlib.Path(r).resolve()) for r in refs))
    return args


def gen(prompt, out_file, refs=None, seed=0, size=SIZE):
    """CLI 는 --output_dir 이 없고 '현재 작업 디렉토리/output/sample_N.png' 에 저장한다.
    그래서 cwd 를 임시 디렉토리로 두고 거기서 결과를 회수한다."""
    tmp = pathlib.Path(tempfile.mkdtemp())
    args = build_args(prompt, refs=refs, seed=seed, size=size)
    env = dict(os.environ, PYTHONPATH=str(FLUX_DIR / "src"))

    r = subprocess.run(args, cwd=tmp, env=env, capture_output=True, text=True)
    made = sorted((tmp / "output").glob("*.png")) or sorted(tmp.glob("**/*.png"))

    if r.returncode != 0 or not made:
        print("--- CLI stdout ---"); print(r.stdout[-2000:])
        print("--- CLI stderr ---"); print(r.stderr[-2000:])
        print("--- tmp 내용 ---")
        for q in sorted(tmp.rglob("*")):
            print("   ", q.relative_to(tmp))
        shutil.rmtree(tmp, ignore_errors=True)
        raise RuntimeError(
            f"생성 실패 (returncode={r.returncode}, png {len(made)}개). "
            "위 stdout 의 인자 오류를 확인할 것."
        )

    out_file.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(made[-1]), out_file)
    shutil.rmtree(tmp, ignore_errors=True)
    return out_file


print("준비됨. 8번 셀(연결 확인)에서 한 장만 먼저 뽑아볼 것.")


## 8. 연결 확인

**전체를 돌리기 전에 한 장만.** 인자가 틀리면 여기서 걸린다.


In [ ]:
test = gen("a plain wooden table in an empty room, soft daylight, no text",
           out_path("_smoke_test", "_smoke_test.png"), seed=0, size=512)
print("생성됨:", test)

from IPython.display import display
from PIL import Image
display(Image.open(test))


## 9. World Reference

**반드시 먼저.** 이걸 레퍼런스로 걸어야 장소끼리 같은 세계로 보인다.


In [ ]:
WR = {}
for scen, v in P.items():
    WR[scen] = gen(v["world"], out_path(scen, "world_ref.png"), seed=0)
    print("world_ref:", scen)


## 10. 장소 배경


In [ ]:
for scen, v in P.items():
    for pid, prompt in v["places"].items():
        # refs=[WR[scen]] 뺐다 — world_ref 를 레퍼런스로 물리면 klein
        # (guidance=1.0) 이 텍스트를 거의 무시하고 레퍼런스를 그대로
        # 복사한다(장소마다 다 같은 그림이 나옴). 텍스트만으로 생성.
        gen(prompt, out_path(scen, "places", f"{pid}.png"), seed=0)
        print("  ", scen, pid)


## 11. 인물 초상 + 표정

베이스 1장을 고정하고 편집으로 표정을 바꾼다.
표정마다 새로 생성하면 매번 얼굴이 달라져 같은 인물로 안 보인다.


In [ ]:
# 표정 세트는 만들지 않는다. 인물당 1장(p0)만 만든다.
# refs=[WR[scen]] 뺐다 — 이유는 장소 루프와 동일(레퍼런스 그대로 복사 문제).
for scen, v in P.items():
    for cid, prompt in v["portraits"].items():
        gen(prompt, out_path(scen, "portraits", f"{cid}_p0.png"), seed=0)
        print("  ", scen, cid)

## 12. 합성 확인

`home_place_id` 로 짝을 맞춰 배경 위에 인물을 얹는다.
모델 선택 기준이 "붙여놨을 때 한 그림인가"이므로 이걸 눈으로 봐야 한다.


In [ ]:
from IPython.display import display
from PIL import Image

def composite(scen, w=900):
    d = DATA[scen]
    for c in d["cast"]:
        bg = OUT_DIR / scen / "places" / f"{c['home_place_id']}.png"
        ch = OUT_DIR / scen / "portraits" / f"{c['id']}_p0.png"
        if not (bg.exists() and ch.exists()):
            continue
        b = Image.open(bg).convert("RGB").resize((w, int(w * 0.6)))
        s = int(b.height * 0.85)
        b.paste(Image.open(ch).convert("RGB").resize((s, s)), (int(w * 0.06), b.height - s))
        out = out_path(scen, "composites", f"{c['id']}.png")
        b.save(out)
        print(f"{c['id']} ({c['occupation']}) @ {c['home_place_id']}")
        display(b)

# 시나리오 하나만 골라 눈으로 확인한다 (전부 다 볼 필요는 없다).
demo_scen = next(iter(DATA))
composite(demo_scen)


## 13. HiDream과 비교

같은 프롬프트로 만든 두 모델의 결과를 나란히 본다.


In [ ]:
from IPython.display import display
from PIL import Image

HIDREAM_DIR = BASE / "outputs" / "image_hidream"

def side_by_side(scen, rel, w=440):
    imgs = []
    for label, root in [("FLUX.2 klein", OUT_DIR), ("HiDream-O1", HIDREAM_DIR)]:
        f = root / scen / rel
        if f.exists():
            imgs.append((label, Image.open(f).convert("RGB")))
        else:
            print("없음:", label, f)
    if not imgs:
        return
    h = int(w * imgs[0][1].height / imgs[0][1].width)
    canvas = Image.new("RGB", (w * len(imgs), h), "white")
    for i, (_, im) in enumerate(imgs):
        canvas.paste(im.resize((w, h)), (i * w, 0))
    print(" | ".join(l for l, _ in imgs), "-", rel)
    display(canvas)

demo_scen = next(iter(DATA))
demo_place = DATA[demo_scen]["places"][0]["place_id"]
demo_cast = DATA[demo_scen]["cast"][0]["id"]
side_by_side(demo_scen, "world_ref.png")
side_by_side(demo_scen, f"places/{demo_place}.png")
side_by_side(demo_scen, f"portraits/{demo_cast}_p0.png")


## 14. 확인 항목

1. 장소끼리 같은 세계로 보이는가 (World Reference가 앵커 역할을 했는가)
2. **시대·문화가 맞는가** — 흥부놀부가 조선인가, 중국/일본풍으로 뭉개졌는가
3. **심문 UI에 얹었을 때 대사가 읽히는가** — 배경이 화려할수록 불리하다
4. 원작 IP 디자인이 새어나오지 않았는가
5. HiDream 과 비교할 때, 여기는 world_ref 를 레퍼런스 이미지로 걸 수 있고
   HiDream 은 문장으로만 맞춘다. 일관성 차이는 이 조건 차이를 감안할 것.